<a href="https://colab.research.google.com/github/lunaliu226/Algorithmic-equity-index-builder/blob/main/IndexCons.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# 1. Install and Import Required Libraries
!pip install yfinance -q
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 2. Define the Initial Asset Universe (AI Sector Focus)
tickers = ['NVDA', 'MSFT', 'GOOGL', 'AMD', 'TSM', 'META', 'PLTR', 'SNOW', 'CRWD', 'AI']
end_date = datetime.today().strftime('%Y-%m-%d')
start_date = (datetime.today() - timedelta(days=365)).strftime('%Y-%m-%d')

print("Status: Ingesting empirical market data...")
# 3. Data Ingestion: Fetch Close prices (auto-adjusted by default)
data = yf.download(tickers, start=start_date, end=end_date)['Close']

print(f"Status: Data successfully extracted. Shape: {data.shape}")

# 4. Calculate Quantitative Metrics
print("Status: Calculating quantitative metrics...")
daily_returns = data.pct_change().dropna()
volatility = daily_returns.std() * np.sqrt(252)
momentum = (data.iloc[-1] / data.iloc[0]) - 1

diagnostics_df = pd.DataFrame({'Momentum': momentum, 'Volatility': volatility})

# 5. Algorithmic Filtration Protocol
print("Status: Executing filtration protocol...")
# Rule 1: Eliminate assets in the top 20% of volatility (structural outlier removal)
vol_threshold = diagnostics_df['Volatility'].quantile(0.80)
filtered_df = diagnostics_df[diagnostics_df['Volatility'] <= vol_threshold].copy()

# Rule 2: Eliminate assets with negative 1-year momentum
filtered_df = filtered_df[filtered_df['Momentum'] > 0].copy()

# 6. Weighting Optimization (Inverse Volatility Weighting)
print("Status: Calculating optimal multi-factor weightings...")
filtered_df['Inverse_Vol'] = 1 / filtered_df['Volatility']
filtered_df['Weight (%)'] = (filtered_df['Inverse_Vol'] / filtered_df['Inverse_Vol'].sum()) * 100

# 7. Final Output Generation
final_index = filtered_df.sort_values(by='Weight (%)', ascending=False)
final_index = final_index[['Momentum', 'Volatility', 'Weight (%)']].round(4)

print("\n--- Final Index Constituents & Calculated Weights ---")
print(final_index)


/tmp/ipykernel_14395/919222284.py:15: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start=start_date, end=end_date)['Close']
[**********            20%                       ]  2 of 10 completed

Status: Ingesting empirical market data...


[*********************100%***********************]  10 of 10 completed

Status: Data successfully extracted. Shape: (250, 10)
Status: Calculating quantitative metrics...
Status: Executing filtration protocol...
Status: Calculating optimal multi-factor weightings...

--- Final Index Constituents & Calculated Weights ---
        Momentum  Volatility  Weight (%)
Ticker                                  
MSFT      0.1863      0.2442     18.3151
GOOGL     1.3218      0.2860     15.6413
NVDA      1.0816      0.3380     13.2362
TSM       1.5368      0.3499     12.7857
META      0.4251      0.3556     12.5783
CRWD      0.1689      0.4264     10.4919
SNOW      0.0509      0.5230      8.5532
PLTR      0.6122      0.5326      8.3983
